# Implementing and Testing GraSP

Paper: [Picking Winning Tickets Before Training by Preserving Gradient Flow](https://arxiv.org/pdf/2002.07376)

In [1]:
import torch
import torch.func as fc
from torch.func import jvp, grad
from torch.autograd.functional import hessian
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

from tqdm import tqdm

from import_shelf import shelf
from shelf.models.transformer import VisionTransformer
from shelf.models.resnet.etc import resnet20
from shelf.dataloaders.cifar import get_CIFAR10_dataset
from shelf.trainers.classic import train, validate
from shelf.trainers.zeroth_order import gradient_fwd, group_by_gradient_exp

/home/cjpark/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Warmup: Calculating $\mathbf{J^\top H J}$

In [2]:
def vthvp(f, primals, tangents):
    def jvp_first(_primals):
        return jvp(f, _primals, tangents)[1]
    
    return jvp(jvp_first, (primals,), (tangents,))[1]

def hvp(f, primals, tangents):
    return jvp(grad(f), primals, tangents)[1]

def f(x):
    return x @ x

x = torch.randn(2048)
tangent = torch.randn(2048)

hessian_value = hessian(f, (x,))[0][0]
result_v_hessian_v = torch.dot(tangent, hessian_value @ tangent)
result_jvp = jvp(f, (x,), (tangent,))[1]
result_vthvp = vthvp(f, (x,), (tangent,))
result_hvp = hvp(f, (x,), (tangent,))
result_vdothvp = torch.dot(tangent, result_hvp)

print(result_jvp)
print(result_vthvp, result_vdothvp, result_v_hessian_v)

newton_point = x - result_jvp / result_vthvp * tangent
newton_jvp = jvp(f, (newton_point,), (tangent,))[1]
print(newton_jvp)

tensor(111.0580)
tensor(4235.5054) tensor(4235.5054) tensor(4235.5054)
tensor(-1.3351e-05)


In [3]:
def functional_xent(
    params,
    buffers,
    names,
    model,
    x,
    t,
):
    y = fc.functional_call(model, ({k: v for k, v in zip(names, params)}, buffers), (x,))
    return F.cross_entropy(y, t)

model = VisionTransformer(
    image_size=32,
    patch_size=4,
    num_classes=10,
    dim=256,
    depth=4,
    heads=6,
    mlp_dim=512,
    dropout=0.1,
    emb_dropout=0.1,
).cuda()

train_loader, test_loader = get_CIFAR10_dataset(batch_size=128)

names = list(model.state_dict().keys())
params = list(model.parameters())
buffers = {}



Files already downloaded and verified


In [5]:
log_vthvp = []
log_vdothvp = []
log_vthvp_newton = []
log_vdothvp_newton = []

with torch.no_grad():

    for x, t in tqdm(train_loader):
        x, t = x.cuda(), t.cuda()

        get_loss_with_params = partial(functional_xent, buffers=buffers, names=names, model=model, x=x, t=t)

        tangent = [torch.randn_like(p) for p in params]

        result_jvp = jvp(get_loss_with_params, (params,), (tangent,))[1]
        result_vthvp = vthvp(get_loss_with_params, (params,), (tangent,))
        result_hvp = hvp(get_loss_with_params, (params,), (tangent,))
        
        tangent_flat = torch.cat([t.view(-1) for t in tangent])
        hvp_flat = torch.cat([h.view(-1) for h in result_hvp])
        result_vdothvp = torch.dot(tangent_flat, hvp_flat)

        log_vthvp.append(result_vthvp)
        log_vdothvp.append(result_vdothvp)

        newton_point_1 = [p - result_jvp / result_vdothvp * t for p, t in zip(params, tangent)]
        newton_point_2 = [p - result_jvp / result_vthvp * t for p, t in zip(params, tangent)]
        newton_jvp_1 = jvp(get_loss_with_params, (newton_point_1,), (tangent,))[1]
        newton_jvp_2 = jvp(get_loss_with_params, (newton_point_2,), (tangent,))[1]
        
        log_vthvp_newton.append(newton_jvp_1)
        log_vdothvp_newton.append(newton_jvp_2)

# avg difference between vthvp and vdothvp
print(torch.mean(torch.abs(torch.tensor(log_vthvp) - torch.tensor(log_vdothvp))))

# avg of newton jvp
print(torch.mean(torch.tensor(log_vthvp_newton).abs()))
print(torch.mean(torch.tensor(log_vdothvp_newton)).abs())

# both of them are close to zero.

100%|██████████| 391/391 [01:02<00:00,  6.30it/s]


tensor(39.4704)
tensor(12.7487)
tensor(35.8497)


## Implementing GraSP

In [9]:
def functional_xent(
    params,
    buffers,
    names,
    model,
    x,
    t,
):
    y = fc.functional_call(model, ({k: v for k, v in zip(names, params)}, buffers), (x,))
    return F.cross_entropy(y, t)

model = VisionTransformer(
    image_size=32,
    patch_size=4,
    num_classes=10,
    dim=256,
    depth=4,
    heads=6,
    mlp_dim=512,
    dropout=0.1,
    emb_dropout=0.1,
).cuda()
# model = resnet20().cuda()

train_loader, test_loader = get_CIFAR10_dataset(batch_size=128)

names = list(model.state_dict().keys())
params = list(model.parameters())
buffers = {}

for name, param in zip(names, params):
    if torch.sum(param) == 0:
        # initialize
        param.data += torch.randn_like(param) * 1e-5


Files already downloaded and verified


In [10]:
with torch.no_grad():

    grasp_score_sum = [torch.zeros_like(p) for p in params]

    for x, t in tqdm(train_loader):
        x, t = x.cuda(), t.cuda()

        get_loss_with_params = partial(functional_xent, buffers=buffers, names=names, model=model, x=x, t=t)

        tangent = [torch.randn_like(p) for p in params]

        result_hvp = hvp(get_loss_with_params, (params,), (tangent,))
        print(result_hvp)
        
        grasp_score = [-p * h for p, h in zip(params, result_hvp)]
        grasp_score_sum = [g + gs for g, gs in zip(grasp_score_sum, grasp_score)]

        break
        

print(grasp_score_sum)

  0%|          | 0/391 [00:00<?, ?it/s]

[tensor([[[-5.0380e-03,  3.3792e-02, -1.1518e-01,  ...,  1.7617e-01,
           3.3352e-01, -5.4967e-02],
         [ 1.4688e-03,  2.3385e-03,  5.7200e-04,  ..., -1.5553e-03,
          -2.2223e-04,  4.5893e-04],
         [ 2.0300e-03,  2.1618e-03,  2.6030e-03,  ..., -4.2726e-04,
           1.3272e-03, -9.6577e-05],
         ...,
         [ 2.8865e-03,  2.3967e-03,  2.5847e-03,  ..., -1.9001e-03,
           9.1164e-04, -9.0969e-04],
         [ 2.7920e-03, -1.3314e-03,  2.4183e-03,  ..., -1.1855e-03,
           2.3398e-04,  2.7440e-03],
         [ 2.7195e-03,  1.0598e-05, -2.7131e-04,  ..., -1.8480e-03,
           1.8407e-03,  1.0894e-03]]], device='cuda:0'), tensor([[[-5.0380e-03,  3.3792e-02, -1.1518e-01,  3.0850e-01, -1.5375e-01,
           2.0060e-01, -1.2067e-01,  6.6316e-03, -2.5782e-02,  7.7961e-02,
           2.0504e-01,  1.9273e-02, -4.0161e-01,  1.6551e-01,  3.7019e-01,
          -2.4314e-01, -1.3523e-01,  3.7494e-01,  1.1867e-01, -1.2063e-01,
          -1.7667e-01, -2.3395e-01,

In [11]:
@torch.no_grad()
def fwd_grasp_score(input, label, model, tangent):
    names = list(model.state_dict().keys())
    params = list(model.parameters())
    buffers = {}

    get_loss_with_params = partial(functional_xent, buffers=buffers, names=names, model=model, x=input, t=label)

    def get_fwd_gradient(params):
        jvp_value = jvp(get_loss_with_params, (params,), (tangent,))[1]
        return [jvp_value * t for t in tangent]
    
    fwd_Hg = jvp(get_fwd_gradient, (params,), (tangent,))[1]

    grasp_score = [-p * h for p, h in zip(params, fwd_Hg)]

    return grasp_score

@torch.no_grad()
def grasp_score(input, label, model):
    names = list(model.state_dict().keys())
    params = list(model.parameters())
    buffers = {}

    get_loss_with_params = partial(functional_xent, buffers=buffers, names=names, model=model, x=input, t=label)

    tangent = grad(get_loss_with_params)(params)

    hvp_value = hvp(get_loss_with_params, (params,), (tangent,))

    grasp_score = [-p * h for p, h in zip(params, hvp_value)]

    return grasp_score

    
x, t = next(iter(train_loader))
x, t = x.cuda(), t.cuda()

tangent = [torch.randn_like(p) for p in params]
gradient = grad(get_loss_with_params)(params)

fwd_grasp = fwd_grasp_score(x, t, model, gradient)
grasp = grasp_score(x, t, model)

# compare the two
fwd_grasp_flat = torch.cat([g.view(-1) for g in fwd_grasp])
grasp_flat = torch.cat([g.view(-1) for g in grasp])

cossim = torch.dot(fwd_grasp_flat, grasp_flat) / (torch.norm(fwd_grasp_flat) * torch.norm(grasp_flat))

print(cossim)
print(torch.norm(fwd_grasp_flat), torch.norm(grasp_flat))

tensor(0.8966, device='cuda:0')
tensor(78.3293, device='cuda:0') tensor(7.2277, device='cuda:0')
